In [ ]:
# Cell 1: Enhanced Imports and Setup for High-Precision GPS Correction
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV
import glob
import os
from geopy.distance import geodesic
import joblib
# Enhanced feature engineering
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_regression
# Additional imports for mapping and visualization
import folium
from folium import plugins
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
# Coordinate transformation
from pyproj import Transformer
import json
import warnings
warnings.filterwarnings('ignore')

In [13]:
# Cell 2: Discover all train drive/phone sets
base_path = 'smartphone-decimeter-2023/sdc2023/test/'
drive_sets = []
for drive in glob.glob(os.path.join(base_path, '*')):
    if not os.path.isdir(drive):
        continue
    for phone in glob.glob(os.path.join(drive, '*')):
        if not os.path.isdir(phone):
            continue
        gnss = os.path.join(phone, 'device_gnss.csv')
        imu = os.path.join(phone, 'device_imu.csv')
        if os.path.exists(gnss) and os.path.exists(imu):
            drive_sets.append({'gnss': gnss, 'imu': imu, 'drive': drive, 'phone': phone})
print(f"Found {len(drive_sets)} drive/phone sets with all required files.")

Found 40 drive/phone sets with all required files.


In [18]:
# Cell 3: Realistic Feature Set Based on Available Data
# These are the features actually available in your dataset
realistic_features = [
    # GNSS signal quality features
    'Cn0DbHz',              # Signal strength (SNR) - CRITICAL for quality
    'Svid',                 # Satellite ID
    'SvElevationDegrees',   # Satellite elevation - CRITICAL for atmospheric errors
    'SvAzimuthDegrees',     # Satellite azimuth
    
    # IMU features  
    'IMU_MessageType',      # IMU message type
    'MeasurementX',         # IMU X measurement
    'MeasurementY',         # IMU Y measurement  
    'MeasurementZ',         # IMU Z measurement
    # "SpeedMps",
    
    # Bias corrections
    'BiasX',                # X bias
    'BiasY',                # Y bias
    'BiasZ',                # Z bias
    
    # WLS ECEF positions (calculated during training, will be calculated for test)
    'WlsPositionXEcefMeters',
    'WlsPositionYEcefMeters', 
    'WlsPositionZEcefMeters'
]

print(f"Realistic feature set: {len(realistic_features)} features")
print("✅ Using only features actually available in your dataset")
print("🎯 Focus: Cn0DbHz (signal strength) + SvElevationDegrees (atmospheric filtering)")
print("📍 Note: WLS ECEF positions calculated during training, will be calculated for test data")

Realistic feature set: 14 features
✅ Using only features actually available in your dataset
🎯 Focus: Cn0DbHz (signal strength) + SvElevationDegrees (atmospheric filtering)
📍 Note: WLS ECEF positions calculated during training, will be calculated for test data


In [19]:
# Cell 4: Find intersection of realistic features across all sets
feature_sets = []
for i, ds in enumerate(drive_sets):
    try:
        gnss = pd.read_csv(ds['gnss'], nrows=1, low_memory=False)
        imu = pd.read_csv(ds['imu'], nrows=1)

        if 'MessageType' in imu.columns:
            imu = imu.rename(columns={'MessageType': 'IMU_MessageType'})

        merged_cols = set(gnss.columns) | set(imu.columns)
        current_features = set([f for f in realistic_features if f in merged_cols])
        feature_sets.append(current_features)

        if i < 5:  # Print first 5 for debugging
            print(f"Set {i} features: {len(current_features)} available")
    except Exception as e:
        print(f"Error reading set {i}: {e}")
        # Use basic features if reading fails
        basic_features = set(['Cn0DbHz', 'Svid', 'SvElevationDegrees', 'SvAzimuthDegrees',
                             'WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters'])
        feature_sets.append(basic_features)


common_features = set.intersection(*feature_sets)
print(f"\nCommon realistic features: {len(list(common_features))}")
print(f"Features: {sorted(list(common_features))}")

Set 0 features: 14 available
Set 1 features: 14 available
Set 2 features: 14 available
Set 3 features: 14 available
Set 4 features: 14 available

Common realistic features: 14
Features: ['BiasX', 'BiasY', 'BiasZ', 'Cn0DbHz', 'IMU_MessageType', 'MeasurementX', 'MeasurementY', 'MeasurementZ', 'SvAzimuthDegrees', 'SvElevationDegrees', 'Svid', 'WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters']


In [25]:
# Cell 5: Enhanced Test Data Loading with Advanced Signal Quality Filtering
from pyproj import Transformer
import numpy as np
import pandas as pd
import os

# Coordinate transformer: ECEF → WGS84
transformer = Transformer.from_crs("EPSG:4978", "EPSG:4326", always_xy=True)

X_all = []
total_samples = 0
high_quality_samples = 0

print(f"Processing {len(drive_sets)} test sets with enhanced filtering...")

for i, drive_set in enumerate(drive_sets):
    drive_name = os.path.basename(drive_set['drive'])
    print(f"Processing set {i+1}/{len(drive_sets)}: {drive_name}")

    try:
        # === Load GNSS + IMU ===
        gnss = pd.read_csv(drive_set['gnss'], low_memory=False)
        imu = pd.read_csv(drive_set['imu'])

        # === Standardize column names ===
        if 'UnixTimeMillis' in gnss.columns:
            gnss.rename(columns={'UnixTimeMillis': 'utcTimeMillis'}, inplace=True)
        if 'UnixTimeMillis' in imu.columns:
            imu.rename(columns={'UnixTimeMillis': 'utcTimeMillis'}, inplace=True)
        if 'MessageType' in imu.columns:
            imu.rename(columns={'MessageType': 'IMU_MessageType'}, inplace=True)

        # === Merge GNSS and IMU by nearest timestamp ===
        gnss_imu = pd.merge_asof(
            gnss.sort_values('utcTimeMillis'),
            imu.sort_values('utcTimeMillis'),
            on='utcTimeMillis',
            direction='nearest',
            tolerance=50
        )
        total_samples += len(gnss_imu)

        # === Enhanced Quality Filters ===
        quality_filters = []

        # 1️⃣ Strong signal (Cn0DbHz > 40)
        if 'Cn0DbHz' in gnss_imu.columns:
            quality_filters.append(gnss_imu['Cn0DbHz'] > 40)

        # 2️⃣ Satellite elevation > 20°
        if 'SvElevationDegrees' in gnss_imu.columns:
            quality_filters.append(gnss_imu['SvElevationDegrees'] > 20)

        # 3️⃣ Valid, non-zero ECEF coordinates
        if all(col in gnss_imu.columns for col in ['WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters']):
            valid_ecef = (
                (gnss_imu['WlsPositionXEcefMeters'] != 0) &
                (gnss_imu['WlsPositionYEcefMeters'] != 0) &
                (gnss_imu['WlsPositionZEcefMeters'] != 0) &
                gnss_imu['WlsPositionXEcefMeters'].notna() &
                gnss_imu['WlsPositionYEcefMeters'].notna() &
                gnss_imu['WlsPositionZEcefMeters'].notna()
            )
            quality_filters.append(valid_ecef)

        # Combine filters
        if quality_filters:
            high_quality_mask = np.logical_and.reduce(quality_filters)
            gnss_imu = gnss_imu[high_quality_mask]
        high_quality_samples += len(gnss_imu)

        if len(gnss_imu) == 0:
            print(f"  ⚠️ Skipped {drive_name} — no high-quality rows after filtering.")
            continue

        # === Convert ECEF → Lat/Lng ===
        if all(col in gnss_imu.columns for col in ['WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters']):
            coords = transformer.transform(
                gnss_imu['WlsPositionXEcefMeters'].values,
                gnss_imu['WlsPositionYEcefMeters'].values,
                gnss_imu['WlsPositionZEcefMeters'].values
            )
            gnss_imu['WlsLatitudeDegrees'] = coords[1]
            gnss_imu['WlsLongitudeDegrees'] = coords[0]

            # Remove invalid or NaN coordinates
            valid_coords = (
                np.isfinite(gnss_imu['WlsLatitudeDegrees']) &
                np.isfinite(gnss_imu['WlsLongitudeDegrees']) &
                (np.abs(gnss_imu['WlsLatitudeDegrees']) <= 90) &
                (np.abs(gnss_imu['WlsLongitudeDegrees']) <= 180)
            )
            gnss_imu = gnss_imu[valid_coords]

        # === Speed Filter (<= 15 m/s ≈ 54 km/h) ===
        if 'SpeedMps' in gnss_imu.columns:
            before_speed = len(gnss_imu)
            gnss_imu = gnss_imu[gnss_imu['SpeedMps'] <= 15.0]
            after_speed = len(gnss_imu)
            print(f"  🚌 Speed filter <= 15 m/s: kept {after_speed}/{before_speed} rows")
            if after_speed == 0:
                print(f"  ⚠️ Skipping {drive_name} — no rows after speed filter.")
                continue
        else:
            print(f"  ⚠️ 'SpeedMps' missing — skipped speed filter for {drive_name}")

        # === Feature Selection ===
        features = [f for f in common_features if f in gnss_imu.columns]
        X = gnss_imu[features].copy()

        # === Derived Features ===
        if 'Cn0DbHz' in X.columns and 'SvElevationDegrees' in X.columns:
            X['SignalQuality'] = X['Cn0DbHz'] * np.sin(np.radians(X['SvElevationDegrees']))
        if 'WlsPositionXEcefMeters' in X.columns:
            X['WLS_Distance'] = np.sqrt(
                X['WlsPositionXEcefMeters']**2 +
                X['WlsPositionYEcefMeters']**2 +
                X['WlsPositionZEcefMeters']**2
            )

        X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

        print(f"  ✅ Added {len(X):,} clean, high-quality rows from {drive_name}")
        X_all.append(X)

    except Exception as e:
        print(f"  ❌ Error processing {drive_name}: {e}")
        continue

# === Combine All Filtered Datasets ===
if len(X_all) > 0:
    X_all_df = pd.concat(X_all, ignore_index=True)
    print(f"\n✅ Combined TEST dataset created with {len(X_all_df):,} total filtered rows.")
else:
    X_all_df = pd.DataFrame()
    print("\n⚠️ No valid data loaded from any test set.")

print(f"\n📊 Summary:")
print(f"   Total raw samples: {total_samples:,}")
print(f"   High-quality samples before speed filter: {high_quality_samples:,}")
print(f"   Final dataset size: {len(X_all_df):,}")


Processing 40 test sets with enhanced filtering...
Processing set 1/40: 2020-12-11-19-30-us-ca-mtv-e
  ⚠️ 'SpeedMps' missing — skipped speed filter for 2020-12-11-19-30-us-ca-mtv-e
  ✅ Added 8,569 clean, high-quality rows from 2020-12-11-19-30-us-ca-mtv-e
Processing set 2/40: 2021-08-17-20-37-us-ca-mtv-g
  ⚠️ 'SpeedMps' missing — skipped speed filter for 2020-12-11-19-30-us-ca-mtv-e
  ✅ Added 8,569 clean, high-quality rows from 2020-12-11-19-30-us-ca-mtv-e
Processing set 2/40: 2021-08-17-20-37-us-ca-mtv-g
  ⚠️ 'SpeedMps' missing — skipped speed filter for 2021-08-17-20-37-us-ca-mtv-g
  ✅ Added 4,735 clean, high-quality rows from 2021-08-17-20-37-us-ca-mtv-g
Processing set 3/40: 2021-08-31-20-37-us-ca-mtv-e
  ⚠️ 'SpeedMps' missing — skipped speed filter for 2021-08-17-20-37-us-ca-mtv-g
  ✅ Added 4,735 clean, high-quality rows from 2021-08-17-20-37-us-ca-mtv-g
Processing set 3/40: 2021-08-31-20-37-us-ca-mtv-e
  ⚠️ 'SpeedMps' missing — skipped speed filter for 2021-08-31-20-37-us-ca-mtv-e

In [26]:
# Cell: Model Inference and Save Predictions
import os
import pandas as pd
import numpy as np
import joblib

print(f"=== Combined TEST Dataset ===")
print(f"Rows: {len(X_all_df):,}")
print(f"Shape: {X_all_df.shape}")

# Load trained objects
encoders = joblib.load("enhanced_label_encoders_v6.pkl")
selected_features = joblib.load("enhanced_features_v6.pkl")
scaler = joblib.load("robust_scaler_v6.pkl")
model = joblib.load("gradient_boosting_model_v6.pkl")

print("\n✅ Loaded pretrained encoders, features, scaler, and model")

# Ensure test data has all required columns
missing_cols = [col for col in selected_features if col not in X_all_df.columns]
if missing_cols:
    print(f"⚠️ Missing columns in test data: {missing_cols}")
    for col in missing_cols:
        X_all_df[col] = 0  # fill missing with zeros

# Align test features
X_test = X_all_df[selected_features].copy()

# Encode categorical columns using pretrained encoders
for col, le in encoders.items():
    if col in X_test.columns:
        X_test[col] = X_test[col].astype(str)
        X_test[col] = le.transform(X_test[col].where(X_test[col].isin(le.classes_), le.classes_[0]))

# Scale numeric features
X_test_scaled = scaler.transform(X_test)

# Predict corrections (model outputs both lat & lng corrections)
y_pred = model.predict(X_test_scaled)

# Wrap into DataFrame
if y_pred.ndim == 2 and y_pred.shape[1] == 2:
    output = X_all_df.copy()
    output['PredictedLatCorrection'] = y_pred[:, 0]
    output['PredictedLngCorrection'] = y_pred[:, 1]
else:
    output = X_all_df.copy()
    output['PredictedCorrection'] = y_pred

# Save predictions
os.makedirs("predictions", exist_ok=True)
output_path = "predictions/test_predictions_v6.csv"
output.to_csv(output_path, index=False)

print(f"\n✅ Predictions saved to: {output_path}")
print(f"Columns in output: {output.columns.tolist()}")


=== Combined TEST Dataset ===
Rows: 289,866
Shape: (289866, 16)

✅ Loaded pretrained encoders, features, scaler, and model
⚠️ Missing columns in test data: ['SpeedMps']

✅ Predictions saved to: predictions/test_predictions_v6.csv
Columns in output: ['WlsPositionZEcefMeters', 'WlsPositionXEcefMeters', 'SvAzimuthDegrees', 'MeasurementZ', 'Cn0DbHz', 'IMU_MessageType', 'SvElevationDegrees', 'BiasY', 'WlsPositionYEcefMeters', 'MeasurementX', 'Svid', 'BiasZ', 'BiasX', 'MeasurementY', 'SignalQuality', 'WLS_Distance', 'SpeedMps', 'PredictedLatCorrection', 'PredictedLngCorrection']

✅ Predictions saved to: predictions/test_predictions_v6.csv
Columns in output: ['WlsPositionZEcefMeters', 'WlsPositionXEcefMeters', 'SvAzimuthDegrees', 'MeasurementZ', 'Cn0DbHz', 'IMU_MessageType', 'SvElevationDegrees', 'BiasY', 'WlsPositionYEcefMeters', 'MeasurementX', 'Svid', 'BiasZ', 'BiasX', 'MeasurementY', 'SignalQuality', 'WLS_Distance', 'SpeedMps', 'PredictedLatCorrection', 'PredictedLngCorrection']
